<img src=images/gdd-logo.png width=300px align=right> 

# Aggregation in Pandas

This section covers all about aggregating with `df.groupby()`, including:

* [Grouped aggregation in pandas](#grouped-agg)
* [Single aggregations with `df.groupby()`](#groupby)
* [Multiple aggregations with `.agg()`](#mult-aggs)
* [<mark>Exercise: Using aggregations</mark>](#ex-aggs)

Let's import pandas and load the data again, since this is a new notebook.

In [1]:
import pandas as pd

chickweight = (
    pd.read_csv('data/chickweight.csv')
    .rename(str.lower, axis='columns')
)

To get statistics for the data you can use methods like:

* `df.describe()`: gives a selection of statistics for each numeric field
* `df.mean()`: calculates the mean of each column or row
* `df.max()`: calculates the max of each column or row
* `df.min()`: calculates the min of each column or row

In [2]:
chickweight.describe()

,rownum,weight,time,chick,diet
count,578.000000,578.000000,578.000000,578.000000,578.000000
mean,289.500000,121.818339,10.717993,25.750865,2.235294
std,166.998503,71.071960,6.758400,14.568795,1.162678
min,1.000000,35.000000,0.000000,1.000000,1.000000
25%,145.250000,63.000000,4.000000,13.000000,1.000000
50%,289.500000,103.000000,10.000000,26.000000,2.000000
75%,433.750000,163.750000,16.000000,38.000000,3.000000
max,578.000000,373.000000,21.000000,50.000000,4.000000


In [3]:
chickweight.mean()

rownum    289.500000
weight    121.818339
time       10.717993
chick      25.750865
diet        2.235294
dtype: float64

<a id = 'grouped-agg'></a>
## Grouped Aggregation in Pandas

However, what if you want data about specific groups of the dataset, e.g. the mean weight for diet 2. In that case you need to ***aggregate*** the data.

Aggregation is the act of splitting up the original dataset to calculate statistics on sub-dataframes.

<img src="images/03_Aggregations/split-groupby-combine.png" width="440" height="440" align="center"/>

There are a few ways to do this in pandas:. 

<a id = 'groupby'></a>
## Single aggregations

To get the overall mean weight you can use:

In [4]:
(
    chickweight
    ['weight']
    .mean()
)

121.81833910034602

But what if you want to see what the mean weight is depending on the diet the chicken was on?

Then you first need to perform a `.groupby()` in order to split the data into those different diets.

In [5]:
(
    chickweight
    .groupby('diet')
    ['weight']
    .mean()
)

diet
1    102.645455
2    122.616667
3    142.950000
4    135.262712
Name: weight, dtype: float64

### <mark>Practice: Try out groupby</mark>

1. Find the minumum `weight` of the chickens at each `time` period.

In [10]:
(
    chickweight
    .groupby(["diet", "time"])
   [["weight"]]
    .min()
)

weight
diet time        
1    0         39
     2         35
     4         48
     6         51
     8         57
     10        51
     12        54
     14        68
     16        71
     18        81
     20        91
     21        96
2    0         39
     2         46
     4         57
     6         72
     8         66
     10        68
     12        70
     14        71
     16        72
     18        72
     20        76
     21        74
3    0         39
     2         48
     4         56
     6         68
     8         80
     10        83
     12       103
     14       112
     16       135
     18       146
     20       156
     21       147
4    0         39
     2         49
     4         61
     6         78
     8         98
     10       117
     12       127
     14       138
     16       145
     18       146
     20       197
     21       196

There are lots of statistics that can be applied to numerical columns, like:
* `.mean()`
* `.sum()`
* `.min()` and `.max()`
* `.first()`
* `.var()`
* `.size()`
* `.count()`

2. Try them out. What does each one do?</mark>

In [20]:
(
    chickweight
    #.sort_values('weight')
    .groupby(["chick"])
   [["weight"]]
    .count()
)

,weight
chick,
1,12
2,12
3,12
4,12
5,12
6,12
7,12
8,11
9,12


In [ ]:
# %load answers/03_Aggregations/practice-aggs.py

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
* `.mean()`: Return the mean of the values 
* `.sum()`: Return the sum of the values 
* `.min()` and `.max()`: Return the minimum or maximum of the values 
* `.first()`: Return the first non-null entry of each column
* `.var()`: Return the variance of each column
* `.size()`: Return the number of rows 
* `.count()`: Return the count of non-null entries of each column

</details>

<mark>**Question:** What happens to the index when you use groupby?</mark>

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
Whatever column(s) you group on will become the new index. 

</details>

You can even perform aggregations when the data is grouped by **multiple** columns:

In [ ]:
(
    chickweight
    .groupby(['diet', 'time'])
    .mean()
)

If grouping by more than one column, the indices will be nested in a _multiindex_, with the order of the list items determining the hierarchy (e.g. `['diet', 'time']` will first group by diet and then within each diet by time).

To take those columns out of the index we can use the `.reset_index()` method.

<mark>**Question**: Why does the below code *not work* when you comment out `.reset_index()`?</mark>

In [26]:
(
    chickweight
    .groupby(['time', 'diet'])
    .mean()
    .reset_index() #macht aus den groupby indexis wieder Spalten
    .loc[lambda df: df['time'] == 10]
)

,time,diet,rownum,weight,chick
20,10,1,112.0,93.052632,10.105263
21,10,2,280.0,108.500000,25.500000
22,10,3,400.0,117.100000,35.500000
23,10,4,518.8,126.000000,45.500000


<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
  Because we are using a filter on the column `time` and when we perform a `.groupby()` on `time` and `diet` these columns become the index of the resulting dataframe. Therefore we cannot filter as we would if they were *normal* columns. 
    
   When you use `.reset_index()` it resets the time and diet as columns and therefore they can be used in the filter.
    
    
   Sometimes this is fine, but sometime you might want to undo this operation by resetting the index. You can use `.reset_index()` at the end to do this.

   The index of a dataframe typically has different behavior than a column.

</details>

<a id = 'mult-aggs'></a>
## Multiple aggregations with `.agg()`

You can get the results for more than one aggregation by using a keyword for the name of the **output column** and a tuple to specificy the **old columns** and the **aggregator** to use. 

```python
df
.groupby('grouping_col')
.agg(output_col = ('old_col', 'aggregator'))
```

**Example**: Find the size of `chick` and mean value for `weight` when grouped by `time`

In [27]:
(
    chickweight
    .groupby('time')
    .agg(num_chickens = ('chick', 'size'), 
         weight_mean = ('weight', 'mean'),
        )
)

,num_chickens,weight_mean
time,,
0,50,41.060000
2,50,49.220000
4,49,59.959184
6,49,74.306122
8,49,91.244898
10,49,107.836735
12,49,129.244898
14,48,143.812500
16,47,168.085106


Note: this way of aggregating became available in `pandas > 0.25`. To check your pandas version you can run the following cell:

In [ ]:
pd.__version__

<a id = 'ex-aggs'></a>
### <mark>Exercise: Using aggregations</mark>

Determine the following aggregate information **per time period**:

- maximum chick id (use the chick column)
- median weight
- std (standard deviation) of weight

- any extras of your choice

**Bonus**: Once you have performed the `.groupby()`, filter the data to only show when the median weight exceeds `150g`.

In [33]:
(
    chickweight
    .groupby('time')
    .agg(max_chick_id = ('chick', 'max'),
         weight_median = ('weight', 'median'),
         weight_std = ('weight', 'std'),
        )
    .loc[lambda df: df['weight_median'] > 150]
)

,max_chick_id,weight_median,weight_std
time,,,
16,50,169.0,46.904079
18,50,187.0,57.394757
20,50,204.0,66.511708
21,50,205.0,71.510273


In [32]:
# %load answers/03_Aggregations/ex-aggs.py
(
    chickweight
    .groupby('time')
    .agg(max_chick_id = ('chick', 'max'),
         weight_median = ('weight', 'median'),
         weight_std = ('weight', 'std'),
        )
    .loc[lambda df: df['weight_median'] > 150]
)

# Conclusion

You have now seen how to do a single aggregation, for example:

```python
(
    df
    .groupby('column')
    .mean()
)
```

And also that you can do multiple aggregations, with control over the output column name, for example:

```python
(
    df
    .groupby('column')
    .agg(min_col = ('old_col', 'min'),
         max_col = ('col_col', 'max')
        )
)
```

An important thing to note is that the pandas GroupBy object is **not a DataFrame until you perform some kind of aggregation function**. 

Once it is a dataframe, you can perform any DataFrame method, for example performing `.loc[]`.